In [8]:
import rdflib
import pandas as pd
import re
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split

from pykeen.pipeline import pipeline

In [14]:
fname = "OWL2RL-10.owl"
g = rdflib.Graph()
g.parse(fname, format='xml')

facts = []
type_triples = []

# Useful namespaces and predicates
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"
schema_namespaces = [
    "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "http://www.w3.org/2000/01/rdf-schema#",
    "http://www.w3.org/2002/07/owl#"
]

# The data properties we want to keep and turn into categories
GOOD_DATA_PROPS = {
    "http://benchmark/OWL2Bench#hasAge",
    "http://benchmark/OWL2Bench#hasResearchInterest",
    "http://benchmark/OWL2Bench#hasTitle",
    "http://benchmark/OWL2Bench#hasPublicationDate"
}

print("Extracting triples...")
for s, p, o in g:
    s_uri, p_uri = str(s), str(p)

    # --- NEW: Intercept rdf:type triples ---
    if p_uri == RDF_TYPE:
        o_uri = str(o)
        # We only want instance-level types (e.g., Person1 type Student).
        # We skip schema-level types (e.g., Student type owl:Class).
        if not any(o_uri.startswith(ns) for ns in ["http://www.w3.org/2000/01/rdf-schema#", "http://www.w3.org/2002/07/owl#"]):
            type_triples.append((s_uri, p_uri, o_uri))
        continue
    # Filter out T-Box (schema) triples, but keep rdf:type for class membership
    if any(p_uri.startswith(ns) for ns in schema_namespaces):
        continue

    # Process Object Properties (Entity -> Relation -> Entity)
    if isinstance(o, (rdflib.term.URIRef, rdflib.term.BNode)):
        facts.append((s_uri, p_uri, str(o)))

    # Process Data Properties (Entity -> Relation -> Literal)
    elif isinstance(o, rdflib.term.Literal):
        # Drop it if it's not in our 'good' list (e.g., Names, IDs, Emails)
        if p_uri not in GOOD_DATA_PROPS:
            continue

        val = str(o.value)

        # Transform 'hasAge' into decade buckets
        if "hasAge" in p_uri:
            try:
                age = float(val)
                bucket = f"<Age_Bucket_{int(age // 10 * 10)}>"
                facts.append((s_uri, p_uri, bucket))
            except ValueError:
                pass # Skip if age is somehow not a number

        # Transform 'hasPublicationDate' into a Year bucket
        elif "hasPublicationDate" in p_uri:
            # Extract the 4-digit year using regex
            match = re.search(r'\d{4}', val)
            if match:
                year = match.group(0)
                facts.append((s_uri, p_uri, f"<Year_{year}>"))

        # Transform 'hasResearchInterest' and 'hasTitle' into categorical URIs
        elif "hasResearchInterest" in p_uri or "hasTitle" in p_uri:
            # Strip the base URL from the predicate for cleaner naming
            prop_name = p_uri.split('#')[-1]
            # Replace spaces with underscores to create a single categorical token
            cat_entity = f"<{prop_name}_{val.replace(' ', '_')}>"
            facts.append((s_uri, p_uri, cat_entity))

print(f"Saving {len(type_triples)} rdf:type triples to entity_types.nt...")
with open("entity_types.nt", "w", encoding="utf-8") as f:
    for s, p, o in type_triples:
        # Standard N-Triples format: <subject> <predicate> <object> .
        f.write(f"<{s}> <{p}> <{o}> .\n")

# 2. Convert to DataFrame
df = pd.DataFrame(facts, columns=['head', 'relation', 'tail'])
print(f"Extracted {len(df)} valid facts.")

# 3. Create ID Mappings
entities = pd.concat([df['head'], df['tail']]).unique()
relations = df['relation'].unique()

# # Map string URIs to integer IDs
# df['head'] = df['head'].map(entity2id)
# df['tail'] = df['tail'].map(entity2id)
# df['relation'] = df['relation'].map(relation2id)

# 4. Split into Train, Valid, Test sets
print("Splitting datasets...")
train_df, temp_df = train_test_split(df, test_size=0.05, random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Save to TSV format (standard for KGE libraries)
train_df.to_csv("OWL2Bench_train.tsv", sep='\t', index=False, header=False)
valid_df.to_csv("OWL2Bench_valid.tsv", sep='\t', index=False, header=False)
test_df.to_csv("OWL2Bench_test.tsv", sep='\t', index=False, header=False)

print("Pipeline complete!")
print(f"Final Vocab - Entities: {len(entities)} | Relations: {len(relations)}")

Extracting triples...
Saving 54047 rdf:type triples to entity_types.nt...
Extracted 406854 valid facts.
Splitting datasets...
Pipeline complete!
Final Vocab - Entities: 50274 | Relations: 39


In [15]:
### full graph to nt
g.serialize("OWL2Bench_full_graph.nt", format="nt")

/Users/thezamp/miniconda3/envs/calibration/lib/python3.10/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


<Graph identifier=N35a0f1590ee44e3ba25fd98911eeb770 (<class 'rdflib.graph.Graph'>)>

In [6]:
result = pipeline(
    training='train.tsv',
    validation='valid.tsv',
    testing='test.tsv',
    model='TransE',    # Good baseline model (others: RotatE, ComplEx)
    device='mps',
    training_kwargs=dict(num_epochs=100),
)

# Output evaluation metrics (MRR, Hits@10)
print(result.metric_results.to_df())

INFO:pykeen.pipeline.api:Using device: mps
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
/Users/thezamp/miniconda3/envs/calibration/lib/python3.10/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Training epochs on mps:0:   0%|          | 0/100 [00:00<?, ?epoch/s]
Training batches on mps:0:   0%|          | 0.00/170 [00:00<?, ?batch/s]
Training batches on mps:0:   1%|          | 1.00/170 [00:00<00:19, 8.62batch/s]
Training batches on mps:0:  15%|█▍        | 25.0/170 [00:00<00:01, 134batch/s] 
Training batches on mps:0:  31%|███       | 52.0/170 [00:00<00:00, 191batch/s]
Training batches on mps:0:  44%|████▍     | 75.0/170 [00:00<00:00, 204batch/s]
Training batches on mps:0:  56%|█████▋    | 96.0/170 [00:00<00:00, 192batch/s]
Training epochs on mps:0:  

     Side    Rank_type                 Metric      Value
0    head   optimistic  z_geometric_mean_rank  62.706528
1    tail   optimistic  z_geometric_mean_rank  64.101145
2    both   optimistic  z_geometric_mean_rank  90.091234
3    head    realistic  z_geometric_mean_rank  62.706525
4    tail    realistic  z_geometric_mean_rank  64.101145
..    ...          ...                    ...        ...
220  tail    realistic     adjusted_hits_at_k   0.482647
221  both    realistic     adjusted_hits_at_k   0.335455
222  head  pessimistic     adjusted_hits_at_k   0.188255
223  tail  pessimistic     adjusted_hits_at_k   0.482647
224  both  pessimistic     adjusted_hits_at_k   0.335455

[225 rows x 4 columns]
